# Colab notebook for Secure Aggregation benchmark

Notebook này clone source, cài dependencies, kiểm tra GPU, thử build CUDA crypto libraries nếu có, rồi chạy benchmark để xuất bảng summary cho paper.

Nếu CUDA libraries chưa build xong, notebook vẫn chạy ở chế độ hybrid / CPU fallback để bạn lấy kết quả trước.

## 1. Mount Drive and clone the repo

Chạy cell này trước để gắn Google Drive và lấy mã nguồn về Colab.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive', force_remount=True)

%cd /content
if not Path('/content/FinalSecureAggregation').exists():
    !git clone https://github.com/quyetnv1611/FinalSecureAggregation.git

%cd /content/FinalSecureAggregation
print('Repo ready at:', Path.cwd())

## 2. Install dependencies and check GPU

Nếu Colab của bạn đang bật T4 hoặc A100, cell này sẽ xác nhận `torch.cuda.is_available()`.

In [ ]:
!apt-get update -y
!apt-get install -y build-essential cmake ninja-build pkg-config git

!python -m pip install -U pip
!python -m pip install -r requirements.txt

import torch
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device:', torch.cuda.get_device_name(0))
else:
    print('No CUDA device detected. Benchmark will fall back to CPU crypto backends unless CUDA libraries are configured.')

## 3. Optional: build or detect CUDA crypto libraries

Cell này sẽ cố gắng build `cuDilithium` từ source và tự tìm `liboqs.so` / `libcuDilithium*.so` trên Drive. Nếu build lỗi, notebook vẫn tiếp tục để bạn có thể chạy hybrid / CPU fallback.

In [ ]:
import os
from pathlib import Path

def find_first(root: str, patterns: list[str]) -> str | None:
    base = Path(root)
    for pattern in patterns:
        for path in base.rglob(pattern):
            if path.is_file():
                return str(path)
    return None

build_dir = Path('/content/drive/MyDrive/secagg_build')
build_dir.mkdir(parents=True, exist_ok=True)

print('Secagg build dir:', build_dir)

# Try building cuDilithium (CUDA SIG) if the source is available
if not Path('/content/cuDilithium').exists():
    !git clone https://github.com/encryptorion-lab/cuDilithium.git /content/cuDilithium

%cd /content/cuDilithium
!rm -rf build || true
!cmake -S . -B build -GNinja -DCMAKE_BUILD_TYPE=Release -DBUILD_SHARED_LIBS=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_CUDA_FLAGS="-Wno-deprecated-gpu-targets" || true
!cmake --build build -j 2 || true
!find build -name 'libcuDilithium*.so' -exec cp -v {} /content/drive/MyDrive/secagg_build/ \; || true

# Try to detect cuPQC SDK on Drive for building liboqs with cuPQC
matches = list(Path('/content/drive').rglob('cuPQCConfig.cmake'))
if matches:
    CUPQC_DIR = str(matches[0].parent)
    print('Found cuPQC SDK at:', CUPQC_DIR)
    %cd /content
    if not Path('/content/liboqs').exists():
        !git clone https://github.com/open-quantum-safe/liboqs.git /content/liboqs
    %cd /content/liboqs
    !rm -rf build || true
    !cmake -S . -B build -GNinja -DOQS_BUILD_ONLY_LIB=ON -DOQS_USE_OPENSSL=OFF -DOQS_USE_CUPQC=ON -DcuPQC_DIR="$CUPQC_DIR" -DCMAKE_BUILD_TYPE=Release || true
    !cmake --build build -j 2 || true
    !find build -name 'liboqs.so' -exec cp -v {} /content/drive/MyDrive/secagg_build/ \; || true
else:
    print('cuPQC SDK not found on Drive — skipping liboqs cuPQC build.')

os.environ['SECAGG_CRYPTO_ACCEL'] = 'cuda'
os.environ['SECAGG_CUDA_LIBRARY_ROOT'] = '/content/drive'

kem_lib = find_first('/content/drive/MyDrive/secagg_build', ['liboqs.so'])
sig_lib = find_first('/content/drive/MyDrive/secagg_build', ['libcuDilithium3.so', 'libcuDilithium2.so', 'libcuDilithium5.so'])

# Also search the whole Drive in case libraries were placed elsewhere
if kem_lib is None:
    kem_lib = find_first('/content/drive', ['liboqs.so'])
if sig_lib is None:
    sig_lib = find_first('/content/drive', ['libcuDilithium3.so', 'libcuDilithium2.so', 'libcuDilithium5.so'])

print('KEM library:', kem_lib)
print('SIG library:', sig_lib)

if kem_lib:
    os.environ['SECAGG_CUDA_KEM_LIBRARY'] = kem_lib
if sig_lib:
    os.environ['SECAGG_CUDA_SIG_LIBRARY'] = sig_lib

if not sig_lib:
    print('No CUDA SIG library found. The benchmark will use CPU fallback for signatures unless you add one to Drive.')
if not kem_lib:
    print('No CUDA KEM library found. The benchmark will use CPU fallback for KEM unless you add one to Drive.')

## 4. Run the benchmark with paper-matching parameters

Bộ tham số này khớp bảng bạn cần: `500` và `1000` clients, dropouts `0% / 10% / 30%`, vector size `100000`, và `10` lần lặp.

In [ ]:
!python experiments/benchmarks/bench_orig_vs_pq.py \
    --crypto-accel cuda \
    --clients 500,1000 \
    --dropouts 0.0,0.1,0.3 \
    --vector-sizes 100000 \
    --n-repeat 10 \
    --reference-clients 500 \
    --reference-dropout 0.0 \
    --reset-checkpoint

## 5. Load the summary table

Cell này lọc đúng phần `clients` để bạn đưa vào paper.

In [ ]:
import pandas as pd
from IPython.display import display

summary_path = Path('results/bench_orig_vs_pq_summary.csv')
if summary_path.exists():
    df = pd.read_csv(summary_path)
    paper_df = df[
        (df['scenario'] == 'clients')
        & (df['n_clients'].isin([500, 1000]))
        & (df['dropout_rate'].isin([0.0, 0.1, 0.3]))
    ].copy()

    show_cols = [
        'n_clients', 'dropout_rate', 'backend_label',
        'advertise_keys_sec', 'share_keys_sec', 'masked_input_sec',
        'unmasking_sec', 'total_time_sec', 'total_comm_mb'
    ]
    available_cols = [col for col in show_cols if col in paper_df.columns]
    display(paper_df[available_cols].sort_values(['n_clients', 'dropout_rate', 'backend_label']))
else:
    print('Summary CSV not found yet. Run the benchmark cell above first.')